# Biohub cell tracking — training walkthrough

The training side of the solution, one step per section:

1. a model trained on the sparse ground truth;
2. pseudo-labels: the whole inference pipeline tracks the training movies;
3. a noisy student fine-tuned on ground truth + pseudo-labels;
4. a second architecture (MultiScaleLineageNet) from scratch on the labels;
5. member-own edge re-scorers (gradient-boosted trees) for the ensemble;
6. inference with the trained ensemble.

`SMOKE = True` runs every step on small synthetic movies with shrunken models in a
few CPU-minutes. With `SMOKE = False` and `DATA` pointing at the competition data
the same calls run the real recipes (GPU, hours per model);
`tools/reproduce_training.sh` chains the full lineage of the shipped models.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

SMOKE = True
DATA = REPO / "outputs" / "walkthrough" / "data"  # <DATA>/train and <DATA>/test
OUT = REPO / "outputs" / "walkthrough"
RECIPES = REPO / "recipes"
RUNS, LABELS, TREES = OUT / "runs", OUT / "labels", OUT / "rescorers"
SMOKE_ARGS = ["--smoke", "--device", "cpu"] if SMOKE else []

## Data

Each movie is `<stem>.zarr` (T x Z x Y x X, uint16; spacing 1.625 x 0.406 x 0.406 um)
with a sparse lineage annotation `<stem>.geff`. The movies in `test/` are pinned to the
test split and never trained on or labelled.

In [ ]:
if SMOKE:
    subprocess.run([sys.executable, str(REPO / "tools" / "make_synthetic_data.py"),
                    "--out", str(DATA), "--movies", "12"], check=True)

from biohub_tracking.training.corpus import load_tracks, movie_stems
from biohub_tracking.training.splits import build_split_from_dir

split = build_split_from_dir(DATA / "train", DATA, seed=314159, fold_test=True)
print(f"train {len(split.train)}, val {len(split.val)}, test {list(split.test)}")
stem = split.train[0]
tracks = load_tracks(DATA / "train", stem)
print(f"{stem}: {tracks.n_nodes} annotated cells, {tracks.n_edges} links, "
      f"{tracks.n_divisions} divisions; organisers' estimate {tracks.estimated_true_nodes:.0f} cells")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import zarr

image = zarr.open_group(DATA / "train" / f"{stem}.zarr", mode="r")["0"]
t = int(np.median(tracks.t))
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(np.asarray(image[t]).max(0), cmap="gray")
points = tracks.zyx[tracks.t == t]
ax.scatter(points[:, 2], points[:, 1], c="red", marker="+", s=80, label="annotated cells")
ax.set_title(f"{stem}, t = {t}, z projection")
ax.legend()
plt.show()

## 1. A model on the ground truth

A recipe (`recipes/train/<run>.json`) holds a run's whole training configuration.
`W-link-s2` is the ground-truth-only model the lineage starts from. Detection
targets are three-state (annotated positive, verified background, unknown with no
loss) plus a count prior from the organisers' per-movie cell-count estimate.

In [ ]:
from biohub_tracking.training import cli as train_cli

common = ["--train-dir", str(DATA / "train"), "--competition-dir", str(DATA),
          "--out-dir", str(RUNS), *SMOKE_ARGS]
train_cli.main(["--recipe", str(RECIPES / "train" / "W-link-s2.json"), "--run-name", "teacher", *common])

history = json.loads((RUNS / "teacher" / "history.json").read_text())
print([round(h["val"]["loss"], 3) for h in history])

## 2. Pseudo-labels from the whole pipeline

The teacher decodes, links and solves every training movie exactly as at
inference. Labels are the ILP's tracks at the decoded (unsmoothed) positions, with
the teacher's probabilities as loss weights; positions are aligned per imaging
cohort to the annotation. An ensemble spec (`recipes/ensembles/*.json`) names the
teacher's models, re-scorers and solver settings.

In [ ]:
from biohub_tracking import labels as labels_cli

def ensemble(name, runs, rescorers=(), smoke_overrides=SMOKE):
    spec = {"runs": list(runs), "member_tta_views": [2] * len(runs) if len(runs) > 1 else [],
            "rescorers": list(rescorers), "overrides": {"solver": {"disappearance_weight": 12.0}}}
    if smoke_overrides:
        # a two-epoch model is barely trained: detect and keep low-confidence cells
        spec["overrides"] = {"detection": {"threshold": 0.01, "tta_views": 2},
                             "solver": {"disappearance_weight": 1.0, "node_logit_bias": -5.0}}
    path = OUT / f"{name}.json"
    path.write_text(json.dumps(spec, indent=1))
    return path

align = [next(s for s in split.train if s.startswith(c)) for c in ("44b6", "6bba")] if SMOKE else None
args = ["--runs", str(RUNS), "--train-dir", str(DATA / "train"), "--competition-dir", str(DATA)]
labels_cli.main(["--ensemble", str(ensemble("teacher", ["teacher"])), *args,
                 "--out", str(LABELS / "round1"), *(["--align-movies", *align] if align else [])])

In [ ]:
from biohub_tracking.training.pseudo import load_pseudo

pseudo = load_pseudo(LABELS / "round1" / f"{stem}.npz")
print(f"{stem}: {len(pseudo.t)} pseudo cells, {int((pseudo.parent >= 0).sum())} pseudo links "
      f"vs {tracks.n_nodes} annotated cells")

## 3. A noisy student

Fine-tuned from the teacher on ground truth + pseudo-labels, with noisy-student
noise: channel dropout, weight decay, shot noise, depth gain and low-contrast haze.
Ground truth always wins the merge; losses are `L_GT + 0.5 * L_pseudo`.

In [ ]:
keep_all = ["--set", "data.pseudo_min_node_prob=0.0", "data.pseudo_min_edge_prob=0.0"] if SMOKE else []
train_cli.main(["--recipe", str(RECIPES / "train" / "R3-ft24-noisy-lc-s1.json"), "--run-name", "student",
                "--init-checkpoint", str(RUNS / "teacher" / "best.pt"),
                "--pseudo-dir", str(LABELS / "round1"), *common, *keep_all])

## 4. A second architecture

`MultiScaleLineageNet` from scratch on the same labels; its linker trains on the
detector's own detections (`proposal_fraction` 0.75).

In [ ]:
train_cli.main(["--recipe", str(RECIPES / "train" / "EX-ms-lc-s5.json"), "--run-name", "multiscale",
                "--pseudo-dir", str(LABELS / "round1"), *common, *keep_all])

## 5. Member-own edge re-scorers

Each model of the ensemble gets its own LightGBM re-ranker of candidate parents,
fitted on its own scores of the shared cells of validation movies (needs
`pip install -e ".[train]"`).

In [ ]:
from biohub_tracking.training import rescorer as rescorer_cli

rescorer_cli.main(["--ensemble", str(ensemble("pair", ["student", "multiscale"])), "--runs", str(RUNS),
                   "--names", "PAIR-m0", "PAIR-m1", "--train-dir", str(DATA / "train"),
                   "--competition-dir", str(DATA), "--out", str(TREES),
                   *(["--movies", *split.val, *align, "--trees", "20"] if SMOKE else [])])

## 6. Inference with the trained ensemble

The two models share one set of detected cells; each scores the candidate links
with its own head and re-scorer; the probabilities are averaged and the ILP solves
the movie. `biohub-predict --ensemble ... --runs ... --rescorers ...` does the same
from the command line.

In [ ]:
from biohub_tracking import cli as predict_cli

settings = OUT / "SETTINGS.json"
settings.write_text(json.dumps({"local": {"TEST_DATA_DIR": str(DATA / "test"), "MODEL_DIR": str(RUNS),
                                          "SUBMISSION_DIR": str(OUT / "submission"),
                                          "WORK_DIR": str(OUT / "work")}}))
pair = ensemble("pair-rescored", ["student", "multiscale"], ["PAIR-m0", "PAIR-m1"])
predict_cli.main(["--settings", str(settings), "--ensemble", str(pair), "--runs", str(RUNS),
                  "--rescorers", str(TREES), "--session-hours", "1"])

## The shipped models

`tools/reproduce_training.sh` runs the full lineage with the real recipes: a
ground-truth model, three rounds of one-model teachers, students on new splits and
architectures, two ensemble teachers (v10, v11), the six final models and their
re-scorers.